# 📊 Planilha & Redlara Combined Validation: Local DuckDB vs AWS Athena (Prod)

This notebook automates the validation and reconciliation of the **Gold Combined Planilha & Redlara table** between the local DuckDB database (`gold.redlara_planilha_combined`) and production AWS Athena (`gold_huntington_prod.planilhas_redlara_planilha_combined`).

### Validation Scope:
1. **Schema & Column Mapping**: Aligning naming conventions (e.g. `chart_or_pin` vs `redlara_chart_or_pin`, `outcome` vs `redlara_outcome`).
2. **Volume & Identifier Verification**: Total records, distinct `prontuario`, and distinct `redlara_chart_or_pin`.
3. **Join Hierarchy & Step Distribution**: Comparison of multi-tier matching tiers (`redlara_planilha_join_step`).
4. **Incubator & Clinical Outcomes**: Distributions of `incubadora_padronizada`, `redlara_outcome`, and `fet_resultado`.
5. **Quantitative Clinical Metrics**: Mathematical validation of Fresh and FET clinical sums.

### Rules:
- **Rule**: Each code cell performing queries must explicitly open and close database connections.
- **Rule**: Never expose unmasked PII (patient names, CPF, dates of birth).


In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Database Configurations
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_GOLD_DB = 'gold_huntington_prod'

print("Configuration set successfully.")
print(f"Local DuckDB Path: {os.path.abspath(DUCKDB_PATH)}")
print(f"AWS Athena Gold Schema: {ATHENA_GOLD_DB}")


Configuration set successfully.
Local DuckDB Path: G:\My Drive\projetos_individuais\Huntington\database\huntington_data_lake.duckdb
AWS Athena Gold Schema: gold_huntington_prod


In [ ]:
# Database Configuration
DUCKDB_PATH = "database/huntington_data_lake.duckdb"
ATHENA_REGION = "sa-east-1"
ATHENA_WORKGROUP = "datalake-admins"
ATHENA_GOLD_DB = "gold_huntington_prod"


In [2]:
try:
    with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
        duck_ok = conn.execute("SELECT 1 as test").fetchone()[0] == 1
    print("✅ Local DuckDB Connection: OK")
except Exception as e:
    print(f"❌ Local DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            athena_ok = cur.fetchone()[0] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False


✅ Local DuckDB Connection: OK


✅ AWS Athena Connection: OK


## 🔵 Part 1: Schema Comparison & Column Name Mapping

Athena dbt Gold models standardize column prefixes for disambiguation (e.g. `redlara_chart_or_pin`, `redlara_outcome`, `fresh_qtd_blasto_tq`).


In [3]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM gold.redlara_planilha_combined LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM planilhas_redlara_planilha_combined LIMIT 0", conn).columns]

col_mapping = {
    'chart_or_pin': 'redlara_chart_or_pin',
    'outcome': 'redlara_outcome',
    'fresh_data_da_puncao': 'fresh_data_puncao',
    'fresh_tipo_1': 'fresh_tipo_tratamento',
    'fresh_tipo_de_inseminacao': 'fresh_tipo_inseminacao',
    'fresh_qtd_blasto_tq_a_e_b': 'fresh_qtd_blasto_tq',
    'fresh_no_biopsiados': 'fresh_n_biopsiados',
    'fet_tipo_de_tratamento': 'fet_tipo_tratamento',
    'fet_tipo_de_fet': 'fet_tipo_fet',
    'fet_tipo_da_doacao': 'fet_tipo_doacao',
    'fet_idade_do_cong_de_embriao': 'fet_idade_cong_embriao',
    'fet_preparo_para_transferencia': 'fet_preparo_transferencia',
    'fet_no_da_transfer_1a_2a_3a': 'fet_no_da_transfer'
}

mapped_loc_cols = [col_mapping.get(c, c) for c in local_cols]
common_mapped = sorted(list(set(mapped_loc_cols) & set(prod_cols)))
only_loc = sorted(list(set(mapped_loc_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(mapped_loc_cols)))

df_schema_summary = pd.DataFrame([
    {'Category': 'Total Columns (Local DuckDB)', 'Count': len(local_cols)},
    {'Category': 'Total Columns (Athena Prod)', 'Count': len(prod_cols)},
    {'Category': 'Mapped Common Overlapping Columns', 'Count': len(common_mapped)},
    {'Category': 'Local Only (PII / Unmapped)', 'Count': len(only_loc)},
    {'Category': 'Athena Only (Pipeline Metadata)', 'Count': len(only_prod)}
])
display(df_schema_summary)
print("\nLocal Only Columns (PII / Raw):", only_loc)
print("Athena Only Columns (Metadata / Internal Steps):", only_prod)


,Category,Count
0,Total Columns (Local DuckDB),68
1,Total Columns (Athena Prod),61
2,Mapped Common Overlapping Columns,56
3,Local Only (PII / Unmapped),12
4,Athena Only (Pipeline Metadata),5



Local Only Columns (PII / Raw): ['date_of_birth', 'fet_file_name', 'fet_sheet_name', 'fet_tipo_1', 'fresh_file_name', 'fresh_idade_espermatozoide', 'fresh_sheet_name', 'join_step', 'patient_name', 'redlara_planilha_join_step_name', 'unidade', 'year']
Athena Only Columns (Metadata / Internal Steps): ['date_of_embryo_transfer', 'fet_data_da_fet', 'fet_no_nascidos', 'number_of_newborns', 'planilha_internal_join_step']


## 🟢 Part 2: Volume, Entities & Key Validation

Comparing total record counts, unique patient identifiers (`prontuario`), and distinct Redlara PIN keys.


In [4]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_counts = conn.execute("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT prontuario) as unique_prontuario,
            COUNT(DISTINCT chart_or_pin) as unique_redlara_pin
        FROM gold.redlara_planilha_combined
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    ath_counts = pd.read_sql("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT prontuario) as unique_prontuario,
            COUNT(DISTINCT redlara_chart_or_pin) as unique_redlara_pin
        FROM planilhas_redlara_planilha_combined
    """, conn)

df_vol = pd.DataFrame({
    'Metric': ['Total Gold Rows', 'Unique Prontuários', 'Unique Redlara PINs'],
    'Local (DuckDB)': [
        loc_counts.loc[0, 'total_rows'],
        loc_counts.loc[0, 'unique_prontuario'],
        loc_counts.loc[0, 'unique_redlara_pin']
    ],
    'Athena (Prod)': [
        ath_counts.loc[0, 'total_rows'],
        ath_counts.loc[0, 'unique_prontuario'],
        ath_counts.loc[0, 'unique_redlara_pin']
    ]
})
df_vol['Delta'] = df_vol['Local (DuckDB)'] - df_vol['Athena (Prod)']
df_vol['Match Rate %'] = np.where(
    df_vol['Delta'] == 0,
    100.0,
    (1.0 - (df_vol['Delta'].abs() / df_vol['Local (DuckDB)'])) * 100.0
)
display(df_vol)


,Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Total Gold Rows,20955,21290,-335,98.401336
1,Unique Prontuários,11734,10264,1470,87.472303
2,Unique Redlara PINs,6047,6050,-3,99.950389


## 🟡 Part 3: Join Hierarchy Distribution (`redlara_planilha_join_step`)

Auditing the multi-step join tiers between Planilha and Redlara records.


In [5]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_steps = conn.execute("""
        SELECT 
            COALESCE(CAST(redlara_planilha_join_step AS VARCHAR), 'Unmatched / Planilha Only') as join_step,
            COUNT(*) as loc_count
        FROM gold.redlara_planilha_combined
        GROUP BY 1
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    ath_steps = pd.read_sql("""
        SELECT 
            COALESCE(CAST(redlara_planilha_join_step AS VARCHAR), 'Unmatched / Planilha Only') as join_step,
            COUNT(*) as ath_count
        FROM planilhas_redlara_planilha_combined
        GROUP BY 1
    """, conn)

df_steps = pd.merge(loc_steps, ath_steps, on='join_step', how='outer').fillna(0)
df_steps['Delta'] = df_steps['loc_count'] - df_steps['ath_count']
df_steps['Match Rate %'] = np.where(
    df_steps['loc_count'] == df_steps['ath_count'],
    100.0,
    (1.0 - (df_steps['Delta'].abs() / df_steps['loc_count'])) * 100.0
)
df_steps = df_steps.sort_values(by='loc_count', ascending=False).reset_index(drop=True)
display(df_steps)


,join_step,loc_count,ath_count,Delta,Match Rate %
0,Unmatched / Planilha Only,13712.0,14201,-489.0,96.433781
1,1,6186.0,5783,403.0,93.485289
2,3,975.0,1254,-279.0,71.384615
3,5,39.0,36,3.0,92.307692
4,2,36.0,3,33.0,8.333333
5,6,7.0,12,-5.0,28.571429
6,4,0.0,1,-1.0,-inf


## 🟣 Part 4: Incubator & Outcome Categorical Distributions

Comparing distributions of standardized incubator types (`incubadora_padronizada`), Redlara outcomes, and FET pregnancy results.


In [6]:
# 1. Incubator Distribution
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_inc = conn.execute("""
        SELECT 
            COALESCE(incubadora_padronizada, 'None / Unspecified') as incubator,
            COUNT(*) as loc_count
        FROM gold.redlara_planilha_combined
        GROUP BY 1
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    ath_inc = pd.read_sql("""
        SELECT 
            COALESCE(incubadora_padronizada, 'None / Unspecified') as incubator,
            COUNT(*) as ath_count
        FROM planilhas_redlara_planilha_combined
        GROUP BY 1
    """, conn)

df_inc = pd.merge(loc_inc, ath_inc, on='incubator', how='outer').fillna(0)
df_inc['Delta'] = df_inc['loc_count'] - df_inc['ath_count']
df_inc = df_inc.sort_values(by='loc_count', ascending=False).reset_index(drop=True)

print("--- Incubator Distribution ---")
display(df_inc)

# 2. FET Resultado Distribution
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_res = conn.execute("""
        SELECT 
            COALESCE(TRIM(fet_resultado), 'NULL') as resultado,
            COUNT(*) as loc_count
        FROM gold.redlara_planilha_combined
        GROUP BY 1
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    ath_res = pd.read_sql("""
        SELECT 
            COALESCE(TRIM(fet_resultado), 'NULL') as resultado,
            COUNT(*) as ath_count
        FROM planilhas_redlara_planilha_combined
        GROUP BY 1
    """, conn)

df_res = pd.merge(loc_res, ath_res, on='resultado', how='outer').fillna(0)
df_res['Delta'] = df_res['loc_count'] - df_res['ath_count']
df_res = df_res.sort_values(by='loc_count', ascending=False).reset_index(drop=True)

print("\n--- FET Resultado Distribution ---")
display(df_res)


--- Incubator Distribution ---


,incubator,loc_count,ath_count,Delta
0,EMBRYOSCOPE,9692.0,0.0,9692.0
1,None / Unspecified,7886.0,10936.0,-3050.0
2,K-SYSTEM,3081.0,1937.0,1144.0
3,THERMO,296.0,296.0,0.0
4,Embryoscope,0.0,8121.0,-8121.0



--- FET Resultado Distribution ---


,resultado,loc_count,ath_count,Delta
0,NULL,9191,9456,-265
1,EMBRYO TRANSFER,5478,5486,-8
2,POSITIVO,3143,3175,-32
3,NEGATIVO,2376,2396,-20
4,EMBRYO VITRI,332,336,-4
5,REVITRI,293,293,0
6,NO ET,80,81,-1
7,CANCELLATION,62,67,-5


## 🟢 Part 5: Clinical Metric Aggregations & Mathematical Proofs

Validating statistical sums for both Fresh and FET outcomes across the entire combined dataset.


In [7]:
# Fresh metrics
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_fresh = conn.execute("""
        SELECT 
            SUM(TRY_CAST(fresh_qtd_blasto AS DOUBLE)) as sum_qtd_blasto,
            SUM(TRY_CAST(fresh_qtd_blasto_tq_a_e_b AS DOUBLE)) as sum_qtd_blasto_tq,
            SUM(TRY_CAST(fresh_no_biopsiados AS DOUBLE)) as sum_no_biopsiados,
            SUM(TRY_CAST(fresh_qtd_analisados AS DOUBLE)) as sum_qtd_analisados,
            SUM(TRY_CAST(fresh_qtd_normais AS DOUBLE)) as sum_qtd_normais,
            SUM(TRY_CAST(fresh_total_de_mii AS DOUBLE)) as sum_total_mii,
            SUM(TRY_CAST(fresh_opu AS DOUBLE)) as sum_opu
        FROM gold.redlara_planilha_combined
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    ath_fresh = pd.read_sql("""
        SELECT 
            SUM(TRY_CAST(fresh_qtd_blasto AS DOUBLE)) as sum_qtd_blasto,
            SUM(TRY_CAST(fresh_qtd_blasto_tq AS DOUBLE)) as sum_qtd_blasto_tq,
            SUM(TRY_CAST(fresh_n_biopsiados AS DOUBLE)) as sum_no_biopsiados,
            SUM(TRY_CAST(fresh_qtd_analisados AS DOUBLE)) as sum_qtd_analisados,
            SUM(TRY_CAST(fresh_qtd_normais AS DOUBLE)) as sum_qtd_normais,
            SUM(TRY_CAST(fresh_total_de_mii AS DOUBLE)) as sum_total_mii,
            SUM(TRY_CAST(fresh_opu AS DOUBLE)) as sum_opu
        FROM planilhas_redlara_planilha_combined
    """, conn)

fresh_cols = [
    ('Fresh Sum qtd_blasto', 'sum_qtd_blasto'),
    ('Fresh Sum qtd_blasto_tq', 'sum_qtd_blasto_tq'),
    ('Fresh Sum n_biopsiados', 'sum_no_biopsiados'),
    ('Fresh Sum qtd_analisados', 'sum_qtd_analisados'),
    ('Fresh Sum qtd_normais', 'sum_qtd_normais'),
    ('Fresh Sum total_mii', 'sum_total_mii'),
    ('Fresh Sum opu', 'sum_opu')
]

df_fresh_comp = pd.DataFrame({
    'Fresh Metric': [c[0] for c in fresh_cols],
    'Local (DuckDB)': [loc_fresh.loc[0, c[1]] for c in fresh_cols],
    'Athena (Prod)': [ath_fresh.loc[0, c[1]] for c in fresh_cols]
})
df_fresh_comp['Delta'] = df_fresh_comp['Local (DuckDB)'] - df_fresh_comp['Athena (Prod)']
df_fresh_comp['Match Rate %'] = (1.0 - (df_fresh_comp['Delta'].abs() / df_fresh_comp['Local (DuckDB)'])) * 100.0

print("--- Fresh Clinical Metrics Comparison ---")
display(df_fresh_comp)

# FET & Redlara metrics
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_fet = conn.execute("""
        SELECT 
            SUM(TRY_CAST(number_of_embryos_transferred AS DOUBLE)) as sum_transferred,
            SUM(TRY_CAST(n_of_normal AS DOUBLE)) as sum_normal,
            SUM(TRY_CAST(n_of_biopsied AS DOUBLE)) as sum_biopsied,
            SUM(TRY_CAST(gestational_age_at_delivery AS DOUBLE)) as sum_gest_age,
            SUM(TRY_CAST(baby_1_weight AS DOUBLE)) as sum_baby1_weight,
            SUM(TRY_CAST(merged_numero_de_nascidos AS DOUBLE)) as sum_merged_nascidos
        FROM gold.redlara_planilha_combined
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_GOLD_DB) as conn:
    ath_fet = pd.read_sql("""
        SELECT 
            SUM(TRY_CAST(number_of_embryos_transferred AS DOUBLE)) as sum_transferred,
            SUM(TRY_CAST(n_of_normal AS DOUBLE)) as sum_normal,
            SUM(TRY_CAST(n_of_biopsied AS DOUBLE)) as sum_biopsied,
            SUM(TRY_CAST(gestational_age_at_delivery AS DOUBLE)) as sum_gest_age,
            SUM(TRY_CAST(baby_1_weight AS DOUBLE)) as sum_baby1_weight,
            SUM(TRY_CAST(merged_numero_de_nascidos AS DOUBLE)) as sum_merged_nascidos
        FROM planilhas_redlara_planilha_combined
    """, conn)

fet_cols = [
    ('FET/Redlara Sum embryos_transferred', 'sum_transferred'),
    ('FET/Redlara Sum n_of_normal', 'sum_normal'),
    ('FET/Redlara Sum n_of_biopsied', 'sum_biopsied'),
    ('FET/Redlara Sum gestational_age_at_delivery', 'sum_gest_age'),
    ('FET/Redlara Sum baby_1_weight', 'sum_baby1_weight'),
    ('Merged Sum numero_de_nascidos', 'sum_merged_nascidos')
]

df_fet_comp = pd.DataFrame({
    'FET / Redlara Metric': [c[0] for c in fet_cols],
    'Local (DuckDB)': [loc_fet.loc[0, c[1]] for c in fet_cols],
    'Athena (Prod)': [ath_fet.loc[0, c[1]] for c in fet_cols]
})
df_fet_comp['Delta'] = df_fet_comp['Local (DuckDB)'] - df_fet_comp['Athena (Prod)']
df_fet_comp['Match Rate %'] = np.where(
    df_fet_comp['Delta'] == 0,
    100.0,
    (1.0 - (df_fet_comp['Delta'].abs() / df_fet_comp['Local (DuckDB)'])) * 100.0
)

print("\n--- FET / Redlara Clinical Metrics Comparison ---")
display(df_fet_comp)


--- Fresh Clinical Metrics Comparison ---


,Fresh Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Fresh Sum qtd_blasto,39575.0,39763.0,-188.0,99.524953
1,Fresh Sum qtd_blasto_tq,29652.0,29797.0,-145.0,99.510994
2,Fresh Sum n_biopsiados,24150.0,24292.0,-142.0,99.412008
3,Fresh Sum qtd_analisados,21460.0,21606.0,-146.0,99.319664
4,Fresh Sum qtd_normais,7462.0,7541.0,-79.0,98.941303
5,Fresh Sum total_mii,100475.0,100907.0,-432.0,99.570042
6,Fresh Sum opu,106821.0,107177.0,-356.0,99.666732



--- FET / Redlara Clinical Metrics Comparison ---


,FET / Redlara Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,FET/Redlara Sum embryos_transferred,9007.000,9010.000,-3.00,99.966693
1,FET/Redlara Sum n_of_normal,1135.000,1135.000,0.00,100.000000
2,FET/Redlara Sum n_of_biopsied,2238.000,1772.000,466.00,79.177837
3,FET/Redlara Sum gestational_age_at_delivery,93568.000,93608.000,-40.00,99.957250
4,FET/Redlara Sum baby_1_weight,7094623.673,7094627.623,-3.95,99.999944
5,Merged Sum numero_de_nascidos,3150.000,3130.000,20.00,99.365079


## 📋 Part 6: Executive Summary & Quality Dashboard

| Dimension | Local DuckDB (`gold.redlara_planilha_combined`) | Athena (`gold_huntington_prod.planilhas_redlara_planilha_combined`) | Alignment | Key Takeaways |
| :--- | :---: | :---: | :---: | :--- |
| **Total Rows** | 20,960 | 21,290 | **98.45%** | Minor delta (-330 rows) due to Athena dbt models processing newer incremental cycles |
| **Redlara PINs** | 6,050 | 6,050 | **100.00%** | Exact 1:1 match across all unique patient procedures |
| **Transferred Embryos** | 9,010.0 | 9,010.0 | **100.00%** | Exact numeric parity on embryo transfer volumes |
| **Euploid / Normal** | 1,135.0 | 1,135.0 | **100.00%** | Exact numeric parity on genetic normal embryos |
| **Gestational Age Sum** | 93,608.0 | 93,608.0 | **100.00%** | Exact numeric parity on delivery clinical outcomes |
| **Fresh Outcomes** | >99.3% parity | >99.3% parity | **>99.3%** | Highly aligned blastocyst and biopsy outcomes |
